# 🏠 Bengaluru House Price Prediction - Exploratory Data Analysis & Advanced ML Pipeline
This notebook cleans the dataset, fixes range parsing bugs, performs feature engineering, and trains a high-accuracy Voting Ensemble model (RandomForest + ExtraTrees).

In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import joblib

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor, VotingRegressor
from sklearn.metrics import r2_score, mean_absolute_error, root_mean_squared_error


In [2]:
# Loading dataset
df = pd.read_csv("Bengaluru_House_Data.csv")
df.head()


,area_type,availability,location,size,society,total_sqft,bath,balcony,price
0,Super built-up Area,19-Dec,Electronic City Phase II,2 BHK,Coomee,1056,2.0,1.0,39.07
1,Plot Area,Ready To Move,Chikka Tirupathi,4 Bedroom,Theanmp,2600,5.0,3.0,120.00
2,Built-up Area,Ready To Move,Uttarahalli,3 BHK,NaN,1440,2.0,3.0,62.00
3,Super built-up Area,Ready To Move,Lingadheeranahalli,3 BHK,Soiewre,1521,3.0,1.0,95.00
4,Super built-up Area,Ready To Move,Kothanur,2 BHK,NaN,1200,2.0,1.0,51.00


In [3]:
# Clean columns
df = df.drop(["area_type", "balcony", "society", "availability"], axis=1, errors="ignore")
df["location"] = df["location"].fillna("Sarjapur  Road").apply(lambda x: str(x).strip())
df["size"] = df["size"].fillna("2 BHK")
df["bath"] = df["bath"].fillna(df["bath"].median()).astype(int)
df["bhk"] = df["size"].apply(lambda x: int(str(x).split()[0]))


In [4]:
# Range parsing bug fix: (a+b)/2
def clean_sqft(sqft):
    parts = str(sqft).split("-")
    try:
        if len(parts) == 2:
            return (float(parts[0].strip()) + float(parts[1].strip())) / 2.0
        return float(parts[0].strip())
    except:
        return np.nan

df["total_sqft"] = df["total_sqft"].apply(clean_sqft)
df = df.dropna(subset=["total_sqft"])
df.drop_duplicates(inplace=True)


In [5]:
# Outlier Filtering
df["price_per_sqft"] = df["price"] * 100000 / df["total_sqft"]
df = df[df["total_sqft"] / df["bhk"] >= 300]
df = df[df["bhk"] <= 6]
df = df[df["bath"] < df["bhk"] + 2]

q1 = df["price_per_sqft"].quantile(0.25)
q3 = df["price_per_sqft"].quantile(0.75)
IQR = q3 - q1
lower_bound = q1 - 0.5 * IQR
upper_bound = q3 + 0.5 * IQR
df = df[(df["price_per_sqft"] >= lower_bound) & (df["price_per_sqft"] <= upper_bound)]
df.reset_index(drop=True, inplace=True)
df[["location", "total_sqft", "bath", "bhk", "price", "price_per_sqft"]].to_csv("cleaned_df.csv", index=False)


In [6]:
# Feature Engineering & Encoding
df["sqft_per_bhk"] = df["total_sqft"] / df["bhk"]
df["bath_per_bhk"] = df["bath"] / df["bhk"]

df_encoded = pd.get_dummies(df.drop(columns=["size", "price_per_sqft"]), columns=["location"], drop_first=True, dtype=int)
X = df_encoded.drop(columns=["price"])
y = df_encoded["price"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


In [7]:
# Train Voting Ensemble
rf = RandomForestRegressor(n_estimators=300, max_depth=22, min_samples_split=3, max_features=0.4, random_state=42, n_jobs=-1)
et = ExtraTreesRegressor(n_estimators=300, max_depth=25, min_samples_split=2, max_features=0.5, random_state=42, n_jobs=-1)
best_model = VotingRegressor([("rf", rf), ("et", et)])
best_model.fit(X_train, y_train)

ypred = best_model.predict(X_test)
print("Testing R2 Score:", r2_score(y_test, ypred))
print("MAE:", mean_absolute_error(y_test, ypred))
joblib.dump(best_model, "rf_model.joblib")
joblib.dump(X_train.columns.tolist(), "model_columns.joblib")


Testing R2 Score: 0.8205105396441172
MAE: 13.263037642018448


['model_columns.joblib']